# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

In [0]:
members_df = spark.sql("select * from members")
bookings_df = spark.sql("select * from bookings")
facilities_df = spark.sql("select * from facilities")

### Question 1

#### How can you produce a list of the start times for bookings by members named 'David Farrell'?

In [0]:
result_df = bookings_df.alias("bks") \
    .join(
        members_df.alias("mems"),
        col("bks.memid") == col("mems.memid"),
        "inner"
    ) \
    .filter(
        (col("mems.firstname") == "David") &
        (col("mems.surname") == "Farrell")
    ) \
    .select(col("bks.starttime"))

display(result_df)

starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z
2012-09-19T15:00:00.000Z
2012-09-19T12:00:00.000Z
2012-09-20T15:30:00.000Z
2012-09-20T11:30:00.000Z
2012-09-20T14:00:00.000Z


### Question 2

#### How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

In [0]:
from pyspark.sql.functions import col, lit

result_df = bookings_df.alias("b") \
    .join(
        facilities_df.alias("f"),
        col("b.facid") == col("f.facid"),
        "inner"
    ) \
    .filter(
        (col("f.name").like("Tennis Court%")) &
        (col("b.starttime") >= lit("2012-09-21")) &
        (col("b.starttime") < lit("2012-09-22"))
    ) \
    .select(
        col("b.starttime"),
        col("f.name")
    ) \
    .orderBy(col("b.starttime"))

result_df.show(truncate=False)

+-------------------+--------------+
|starttime          |name          |
+-------------------+--------------+
|2012-09-21 08:00:00|Tennis Court 2|
|2012-09-21 08:00:00|Tennis Court 1|
|2012-09-21 09:30:00|Tennis Court 1|
|2012-09-21 10:00:00|Tennis Court 2|
|2012-09-21 11:30:00|Tennis Court 2|
|2012-09-21 12:00:00|Tennis Court 1|
|2012-09-21 13:30:00|Tennis Court 1|
|2012-09-21 14:00:00|Tennis Court 2|
|2012-09-21 15:30:00|Tennis Court 1|
|2012-09-21 16:00:00|Tennis Court 2|
|2012-09-21 17:00:00|Tennis Court 1|
|2012-09-21 18:00:00|Tennis Court 2|
+-------------------+--------------+



### Question 3

#### How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).

In [0]:
result_df = members_df.alias("mems") \
    .join(
        members_df.alias("recs"),
        col("mems.recommendedby") == col("recs.memid"),
        "left"
    ) \
    .select(
        col("mems.firstname").alias("memfname"),
        col("mems.surname").alias("memsname"),
        col("recs.firstname").alias("recfname"),
        col("recs.surname").alias("recsname")
    ) \
    .orderBy(col("memsname"), col("memfname"))

result_df.show(truncate=False)

+---------+---------+---------+--------+
|memfname |memsname |recfname |recsname|
+---------+---------+---------+--------+
|Florence |Bader    |Ponder   |Stibbons|
|Anne     |Baker    |Ponder   |Stibbons|
|Timothy  |Baker    |Jemima   |Farrell |
|Tim      |Boothe   |Tim      |Rownam  |
|Gerald   |Butters  |Darren   |Smith   |
|Joan     |Coplin   |Timothy  |Baker   |
|Erica    |Crumpet  |Tracy    |Smith   |
|Nancy    |Dare     |Janice   |Joplette|
|David    |Farrell  |NULL     |NULL    |
|Jemima   |Farrell  |NULL     |NULL    |
|GUEST    |GUEST    |NULL     |NULL    |
|Matthew  |Genting  |Gerald   |Butters |
|John     |Hunt     |Millicent|Purview |
|David    |Jones    |Janice   |Joplette|
|Douglas  |Jones    |David    |Jones   |
|Janice   |Joplette |Darren   |Smith   |
|Anna     |Mackenzie|Darren   |Smith   |
|Charles  |Owen     |Darren   |Smith   |
|David    |Pinker   |Jemima   |Farrell |
|Millicent|Purview  |Tracy    |Smith   |
+---------+---------+---------+--------+
only showing top

### Qeustion 4

#### How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

In [0]:
from pyspark.sql.functions import concat_ws

result_df = bookings_df.alias("b") \
    .join(
        members_df.alias("m"),
        col("b.memid") == col("m.memid"),
        "inner"
    ) \
    .join(
        facilities_df.alias("f"),
        col("b.facid") == col("f.facid"),
        "inner"
    ) \
    .filter(col("f.name").like("Tennis Court%")) \
    .select(
        concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member_name"),
        col("f.name").alias("facility_name")
    ) \
    .distinct() \
    .orderBy(col("member_name"), col("facility_name"))

result_df.show(truncate=False)

+--------------+--------------+
|member_name   |facility_name |
+--------------+--------------+
|Anne Baker    |Tennis Court 1|
|Anne Baker    |Tennis Court 2|
|Burton Tracy  |Tennis Court 1|
|Burton Tracy  |Tennis Court 2|
|Charles Owen  |Tennis Court 1|
|Charles Owen  |Tennis Court 2|
|Darren Smith  |Tennis Court 2|
|David Farrell |Tennis Court 1|
|David Farrell |Tennis Court 2|
|David Jones   |Tennis Court 1|
|David Jones   |Tennis Court 2|
|David Pinker  |Tennis Court 1|
|Douglas Jones |Tennis Court 1|
|Erica Crumpet |Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|GUEST GUEST   |Tennis Court 1|
|GUEST GUEST   |Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows


### Question 5

#### How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

# Create a lookup dictionary: memid -> "firstname surname"
member_lookup = {
    row["memid"]: f"{row['firstname']} {row['surname']}"
    for row in members_df.select("memid", "firstname", "surname").collect()
}

# UDF to get recommender name from recommendedby
def get_recommender_name(recommendedby):
    if recommendedby is None:
        return None
    return member_lookup.get(recommendedby)

get_recommender_name_udf = udf(get_recommender_name, StringType())

result_df = members_df.select(
    concat_ws(" ", col("firstname"), col("surname")).alias("member"),
    get_recommender_name_udf(col("recommendedby")).alias("recommender")
).distinct().orderBy("member")

result_df.show(truncate=False)

+-----------------------+---------------+
|member                 |recommender    |
+-----------------------+---------------+
|Anna Mackenzie         |Darren Smith   |
|Anne Baker             |Ponder Stibbons|
|Burton Tracy           |NULL           |
|Charles Owen           |Darren Smith   |
|Darren Smith           |NULL           |
|David Farrell          |NULL           |
|David Jones            |Janice Joplette|
|David Pinker           |Jemima Farrell |
|Douglas Jones          |David Jones    |
|Erica Crumpet          |Tracy Smith    |
|Florence Bader         |Ponder Stibbons|
|GUEST GUEST            |NULL           |
|Gerald Butters         |Darren Smith   |
|Henrietta Rumney       |Matthew Genting|
|Henry Worthington-Smyth|Tracy Smith    |
|Hyacinth Tupperware    |NULL           |
|Jack Smith             |Darren Smith   |
|Janice Joplette        |Darren Smith   |
|Jemima Farrell         |NULL           |
|Joan Coplin            |Timothy Baker  |
+-----------------------+---------

### Question 6

#### Produce a count of the number of recommendations each member has made. Order by member ID.

In [0]:
from pyspark.sql.functions import count

result_df = members_df.filter(col("recommendedby").isNotNull()) \
    .groupBy("recommendedby") \
    .agg(count("*").alias("recommendation_count")) \
    .orderBy("recommendedby")

result_df.show()

+-------------+--------------------+
|recommendedby|recommendation_count|
+-------------+--------------------+
|            1|                   5|
|            2|                   3|
|            3|                   1|
|            4|                   2|
|            5|                   1|
|            6|                   1|
|            9|                   2|
|           11|                   1|
|           13|                   2|
|           15|                   1|
|           16|                   1|
|           20|                   1|
|           30|                   1|
+-------------+--------------------+



### Question 7

#### Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.

In [0]:
#from pyspark.sql.functions import sum

result_df = bookings_df.groupBy("facid") \
    .agg(sum("slots").alias("total_slots")) \
    .orderBy(col("facid"))

result_df.show()

+-----+-----------+
|facid|total_slots|
+-----+-----------+
|    0|       1320|
|    1|       1278|
|    2|       1209|
|    3|        830|
|    4|       1404|
|    5|        228|
|    6|       1104|
|    7|        908|
|    8|        911|
+-----+-----------+



### Question 8

#### Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

In [0]:
result_df = bookings_df.filter(
    (col("starttime") >= "2012-09-01") &
    (col("starttime") < "2012-10-01")
) \
.groupBy("facid") \
.agg(sum("slots").alias("total_slots")) \
.orderBy(col("total_slots"))

result_df.show()

+-----+-----------+
|facid|total_slots|
+-----+-----------+
|    5|        122|
|    3|        422|
|    7|        426|
|    8|        471|
|    6|        540|
|    2|        570|
|    1|        588|
|    0|        591|
|    4|        648|
+-----+-----------+



### Question 9

#### Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

In [0]:
from pyspark.sql.functions import month, year

result_df = bookings_df.filter(year(col("starttime")) == 2012) \
    .groupBy(
        col("facid"),
        month(col("starttime")).alias("month")
    ) \
    .agg(
        sum("slots").alias("Total Slots")
    ) \
    .orderBy("facid", "month")

result_df.show()

+-----+-----+-----------+
|facid|month|Total Slots|
+-----+-----+-----------+
|    0|    7|        270|
|    0|    8|        459|
|    0|    9|        591|
|    1|    7|        207|
|    1|    8|        483|
|    1|    9|        588|
|    2|    7|        180|
|    2|    8|        459|
|    2|    9|        570|
|    3|    7|        104|
|    3|    8|        304|
|    3|    9|        422|
|    4|    7|        264|
|    4|    8|        492|
|    4|    9|        648|
|    5|    7|         24|
|    5|    8|         82|
|    5|    9|        122|
|    6|    7|        164|
|    6|    8|        400|
+-----+-----+-----------+
only showing top 20 rows


### Question 10

#### Find the total number of members (including guests) who have made at least one booking.

In [0]:
from pyspark.sql.functions import countDistinct

result_df = bookings_df.agg(
    countDistinct("memid").alias("member_count")
)

result_df.show()

+------------+
|member_count|
+------------+
|          30|
+------------+



### Question 11

#### Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.

In [0]:
from pyspark.sql.functions import min

result_df = bookings_df.alias("bks") \
    .join(
        members_df.alias("mems"),
        col("bks.memid") == col("mems.memid"),
        "inner"
    ) \
    .filter(col("bks.starttime") >= "2012-09-01") \
    .groupBy(
        col("mems.surname"),
        col("mems.firstname"),
        col("mems.memid")
    ) \
    .agg(
        min(col("bks.starttime")).alias("starttime")
    ) \
    .orderBy(col("mems.memid"))

result_df.show(truncate=False)

+---------+---------+-----+-------------------+
|surname  |firstname|memid|starttime          |
+---------+---------+-----+-------------------+
|GUEST    |GUEST    |0    |2012-09-01 08:00:00|
|Smith    |Darren   |1    |2012-09-01 09:00:00|
|Smith    |Tracy    |2    |2012-09-01 11:30:00|
|Rownam   |Tim      |3    |2012-09-01 16:00:00|
|Joplette |Janice   |4    |2012-09-01 15:00:00|
|Butters  |Gerald   |5    |2012-09-02 12:30:00|
|Tracy    |Burton   |6    |2012-09-01 15:00:00|
|Dare     |Nancy    |7    |2012-09-01 12:30:00|
|Boothe   |Tim      |8    |2012-09-01 08:30:00|
|Stibbons |Ponder   |9    |2012-09-01 11:00:00|
|Owen     |Charles  |10   |2012-09-01 11:00:00|
|Jones    |David    |11   |2012-09-01 09:30:00|
|Baker    |Anne     |12   |2012-09-01 14:30:00|
|Farrell  |Jemima   |13   |2012-09-01 09:30:00|
|Smith    |Jack     |14   |2012-09-01 11:00:00|
|Bader    |Florence |15   |2012-09-01 10:30:00|
|Baker    |Timothy  |16   |2012-09-01 15:00:00|
|Pinker   |David    |17   |2012-09-01 08

### Question 12

#### Output the names of all members, formatted as 'Surname, Firstname'

In [0]:
from pyspark.sql.functions import col, concat, lit
result_df = members_df.select(
    concat(col("surname"), lit(", "), col("firstname")).alias("member_name")
).orderBy("surname", "firstname")

result_df.show(truncate=False)

+------------------+
|member_name       |
+------------------+
|Bader, Florence   |
|Baker, Anne       |
|Baker, Timothy    |
|Boothe, Tim       |
|Butters, Gerald   |
|Coplin, Joan      |
|Crumpet, Erica    |
|Dare, Nancy       |
|Farrell, David    |
|Farrell, Jemima   |
|GUEST, GUEST      |
|Genting, Matthew  |
|Hunt, John        |
|Jones, David      |
|Jones, Douglas    |
|Joplette, Janice  |
|Mackenzie, Anna   |
|Owen, Charles     |
|Pinker, David     |
|Purview, Millicent|
+------------------+
only showing top 20 rows


### Question 13

#### Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.

In [0]:
from pyspark.sql.functions import lower, col

result_df = facilities_df.filter(
    lower(col("name")).startswith("tennis")
)

result_df.show(truncate=False)

+-----+--------------+----------+---------+-------------+------------------+
|facid|name          |membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+--------------+----------+---------+-------------+------------------+
|0    |Tennis Court 1|5.0       |25.0     |10000.0      |200.0             |
|1    |Tennis Court 2|5.0       |25.0     |8000.0       |200.0             |
+-----+--------------+----------+---------+-------------+------------------+



### Question 14

#### You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.


In [0]:
result_df = members_df.filter(
    col("telephone").like("%(%")
).select(
    "memid",
    "telephone"
).orderBy("memid")

result_df.show(truncate=False)

+-----+--------------+
|memid|telephone     |
+-----+--------------+
|0    |(000) 000-0000|
|3    |(844) 693-0723|
|4    |(833) 942-4710|
|5    |(844) 078-4130|
|6    |(822) 354-9973|
|7    |(833) 776-4001|
|8    |(811) 433-2547|
|9    |(833) 160-3900|
|10   |(855) 542-5251|
|11   |(844) 536-8036|
|13   |(855) 016-0163|
|14   |(822) 163-3254|
|15   |(833) 499-3527|
|20   |(811) 972-1377|
|21   |(822) 661-2898|
|22   |(822) 499-2232|
|24   |(822) 413-1470|
|27   |(822) 989-8876|
|28   |(855) 755-9876|
|29   |(855) 894-3758|
+-----+--------------+
only showing top 20 rows


### Question 15

#### You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

In [0]:
from pyspark.sql.functions import substring, count

result_df = members_df.select(
    substring(col("surname"), 1, 1).alias("first_letter")
).groupBy(
    "first_letter"
).agg(
    count("*").alias("member_count")
).orderBy(
    "first_letter"
)

result_df.show()

+------------+------------+
|first_letter|member_count|
+------------+------------+
|           B|           5|
|           C|           2|
|           D|           1|
|           F|           2|
|           G|           2|
|           H|           1|
|           J|           3|
|           M|           1|
|           O|           1|
|           P|           2|
|           R|           2|
|           S|           6|
|           T|           2|
|           W|           1|
+------------+------------+



### Question 16

#### Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.

In [0]:
from pyspark.sql import functions as F

result_df = spark.range(1).select(
    F.explode(
        F.sequence(
            F.to_date(F.lit("2012-10-01")),
            F.to_date(F.lit("2012-10-31")),
            F.expr("interval 1 day")
        )
    ).alias("october_date")
)

result_df.show(31, truncate=False)

+------------+
|october_date|
+------------+
|2012-10-01  |
|2012-10-02  |
|2012-10-03  |
|2012-10-04  |
|2012-10-05  |
|2012-10-06  |
|2012-10-07  |
|2012-10-08  |
|2012-10-09  |
|2012-10-10  |
|2012-10-11  |
|2012-10-12  |
|2012-10-13  |
|2012-10-14  |
|2012-10-15  |
|2012-10-16  |
|2012-10-17  |
|2012-10-18  |
|2012-10-19  |
|2012-10-20  |
|2012-10-21  |
|2012-10-22  |
|2012-10-23  |
|2012-10-24  |
|2012-10-25  |
|2012-10-26  |
|2012-10-27  |
|2012-10-28  |
|2012-10-29  |
|2012-10-30  |
|2012-10-31  |
+------------+



### Question 17

#### Return a count of bookings for each month, sorted by month

In [0]:
result_df = bookings_df.groupBy(
    month(col("starttime")).alias("month")
).agg(
    count("*").alias("booking_count")
).orderBy("month")

result_df.show()

+-----+-------------+
|month|booking_count|
+-----+-------------+
|    1|            1|
|    7|          658|
|    8|         1472|
|    9|         1913|
+-----+-------------+

